In [ ]:
import requests
session = requests.session()
base_url = "https://maziar-temp-project.ir/connect4"

In [ ]:
#TODO Change the emails to your own email and your opponent email and please go to https://maziarheidari2003.github.io/connect4game and make an account first

my_auth_data = session.post(f"{base_url}/api/v1/auth/login", json={"email":"bot@gmail.com"}).json()
my_id = my_auth_data["id"]
my_token ="Bearer " + my_auth_data["access_token"]

opponent_auth_data = session.post(f"{base_url}/api/v1/auth/login", json={"email":"human@gmail.com"}).json()
opponent_id = opponent_auth_data["id"]
opponent_token = "Bearer " + opponent_auth_data["access_token"]


In [ ]:
#Run this block to create the game
created_game_uuid = session.post(f"{base_url}/api/v1/game/?bot_or_human=TwoSideBot",  headers={"Authorization": my_token}).json()

In [ ]:
#TODO you should replace your code with the random method





import random
import numpy as np

m = 6
n = 7
human = 1  
bot = 2   

matrix = np.zeros((m, n), dtype=int)

def my_make_move(opponent_last_move: int | None = None):
    global matrix
    
    
  
    if opponent_last_move is not None:
        column = opponent_last_move
       
        
        row = rey_get_next_open_row(matrix, column)
        if row is not None:
            matrix[row][column] = human 
            
    
    rey_print_matrix(matrix)
    





    

    col, minimax_score = rey_minimax(matrix, 4, -100000000, 1000000000, True, bot, human)
    
    if col is None or not rey_isValid(matrix, col):
        valid_locations = rey_possibleCols(matrix)
        col = random.choice(valid_locations) if valid_locations else 3



    #if rey_isValid(matrix, col):
     #   for r in range(m):
     #       if matrix[r][col] == 0:
     #        row = r
      #       break
       # matrix[row][col] = bot
   



    return col
    





def rey_print_matrix(matrix):

    flipped = np.flip(matrix, 0)
    print("\nCurrent Board:")
    print("-" * 30)
    for row in flipped:
        print("|", end="")
        for cell in row:
            if cell == 0:
                print(" ◯ ", end="")  # Empty
            elif cell == human:
                print(" 🔴 ", end="")  # Human
            elif cell == bot:
                print(" 🟡 ", end="")  # Bot
        print("|")
    print("-" * 30)
    print("  0   1   2   3   4   5   6")  




def rey_isValid(A, col):
    return A[m-1][col] == 0
    


def rey_possibleCols(A):
    columns = []
    for col in range(n):
        if A[m-1][col] == 0:
            columns.append(col)
    return columns





def rey_minimax(matrix, depth, alpha, beta, maxPlayer, ai_piece, player_piece):
    columns = rey_possibleCols(matrix)
    is_terminal = rey_is_terminal_node(matrix)
    
    if depth == 0 or is_terminal:
        if is_terminal:
            if rey_winning_move(matrix, ai_piece):
                return (None, 100000000)
            elif rey_winning_move(matrix, player_piece):
                return (None, -10000000)
            else:  # Game is over
                return (None, 0)
        else:  # Depth=0
            return (None, rey_counting_cells(matrix, ai_piece))
    
    if maxPlayer:
        value = -np.inf
        column = random.choice(columns)
        
        for col in columns:
            for r in range(m):
                if matrix[r][col] == 0:
                 row = r
                 break

            matrix_copy = matrix.copy()
            matrix_copy[row][col]=ai_piece
            
            new_score = rey_minimax(matrix_copy, depth-1, alpha, beta, False, ai_piece, player_piece)[1]
            
            if new_score > value:
                value = new_score
                column = col
            
            alpha = max(alpha, value)
            if alpha >= beta:
                break
        
        return column, value
    
    else:  # Minplayer
        value = np.inf
        column = random.choice(columns)
        
        for col in columns:
            for r in range(m):
                if matrix[r][col] == 0:
                    row = r
                    break


            matrix_copy = matrix.copy()
            matrix_copy[row][col]=player_piece
            
            new_score = rey_minimax(matrix_copy, depth-1, alpha, beta, True, ai_piece, player_piece)[1]
            
            if new_score < value:
                value = new_score
                column = col
            
            beta = min(beta, value)
            if alpha >= beta:
                break
        
        return column, value







def rey_winning_move(A, piece):
    
    for c in range(n-3):
        for r in range(m):
            if A[r][c] == piece and A[r][c+1] == piece and A[r][c+2] == piece and A[r][c+3] == piece:
                return True

   
    for c in range(n):
        for r in range(m-3):
            if A[r][c] == piece and A[r+1][c] == piece and A[r+2][c] == piece and A[r+3][c] == piece:
                return True

    
    for c in range(n-3):
        for r in range(m-3):
            if A[r][c] == piece and A[r+1][c+1] == piece and A[r+2][c+2] == piece and A[r+3][c+3] == piece:
                return True

   
    for c in range(n-3):
        for r in range(3, m):
            if A[r][c] == piece and A[r-1][c+1] == piece and A[r-2][c+2] == piece and A[r-3][c+3] == piece:
                return True
    
    return False




def rey_computeHuristic(p, o, z):
    huristic = 0
    
    if p == 4: huristic += 10000
    elif p == 3 and z == 1: huristic += 100
    elif p == 2 and z == 2: huristic += 10
    
    
    if o == 3 and z == 1: huristic -= 5000 
    elif o == 2 and z == 2: huristic -= 50
    return huristic






def rey_get_next_open_row(matrix, col):
	for r in range(m):
		if matrix[r][col] == 0:
			return r





def rey_counting_cells(matrix, piece):
    count = 0
    
    if piece == bot:
        opponent = human
    else:
       opponent= bot

    

    
    center_array = matrix[:, n//2]
    count += np.count_nonzero(center_array == piece) * 3

    
    for r in range(m):
        for c in range(n-3):
            p, o, z = 0, 0, 0
            for i in range(4):
                val = matrix[r][c+i]
                if val == piece: p += 1
                elif val == opponent: o += 1
                else: z += 1
            count += rey_computeHuristic(p, o, z)

    
    for c in range(n):
        for r in range(m-3):
            p, o, z = 0, 0, 0
            for i in range(4):
                val = matrix[r+i][c]
                if val == piece: p += 1
                elif val == opponent: o += 1
                else: z += 1
            count += rey_computeHuristic(p, o, z)

   
    for r in range(m-3):
        for c in range(n-3):
            p, o, z = 0, 0, 0
            for i in range(4):
                val = matrix[r+i][c+i]
                if val == piece: p += 1
                elif val == opponent: o += 1
                else: z += 1
            count += rey_computeHuristic(p, o, z)

   
    for r in range(3, m):
        for c in range(n-3):
            p, o, z = 0, 0, 0
            for i in range(4):
                val = matrix[r-i][c+i]
                if val == piece: p += 1
                elif val == opponent: o += 1
                else: z += 1
            count += rey_computeHuristic(p, o, z)

    return count



def rey_is_terminal_node(matrix):
    return rey_winning_move(matrix, human) or rey_winning_move(matrix, bot) or len(rey_possibleCols(matrix)) == 0





In [ ]:
#TODO you should replace your code with the random method



import random
import numpy as np

m = 6
n = 7
human = 1  
bot = 2   

matrix = np.zeros((m, n), dtype=int)

def opponent_make_move(token, game_uuid, opponent_last_move: int | None = None):
    global matrix
    
    
  
    if opponent_last_move is not None:
        column = opponent_last_move
       
        
        row = rey_get_next_open_row(matrix, column)
        if row is not None:
            matrix[row][column] = bot 
            
    
    rey_print_matrix(matrix)
    





    

    col, minimax_score = rey_minimax(matrix, 4, -100000000, 1000000000, True, human, bot)
    
    if col is None or not rey_isValid(matrix, col):
        valid_locations = rey_possibleCols(matrix)
        col = random.choice(valid_locations) if valid_locations else 3
    
    


    #if rey_isValid(matrix, col):
     #   for r in range(m):
     #       if matrix[r][col] == 0:
     #        row = r
      #       break
       # matrix[row][col] = human

    
    res = session.patch(
        f"{base_url}/api/v1/game/make-move",
        params={"game_uuid": game_uuid, "chosen_column": int(col)},
        headers={"Authorization": token}
    )
    return res





def rey_print_matrix(matrix):

    flipped = np.flip(matrix, 0)
    print("\nCurrent Board:")
    print("-" * 30)
    for row in flipped:
        print("|", end="")
        for cell in row:
            if cell == 0:
                print(" ◯ ", end="")  # Empty
            elif cell == human:
                print(" 🔴 ", end="")  # Human
            elif cell == bot:
                print(" 🟡 ", end="")  # Bot
        print("|")
    print("-" * 30)
    print("  0   1   2   3   4   5   6")  




def rey_isValid(A, col):
    return A[m-1][col] == 0
    


def rey_possibleCols(A):
    columns = []
    for col in range(n):
        if A[m-1][col] == 0:
            columns.append(col)
    return columns





def rey_minimax(matrix, depth, alpha, beta, maxPlayer, ai_piece, player_piece):
    columns = rey_possibleCols(matrix)
    is_terminal = rey_is_terminal_node(matrix)
    
    if depth == 0 or is_terminal:
        if is_terminal:
            if rey_winning_move(matrix, ai_piece):
                return (None, 100000000)
            elif rey_winning_move(matrix, player_piece):
                return (None, -10000000)
            else:  # Game is over
                return (None, 0)
        else:  # Depth=0
            return (None, rey_counting_cells(matrix, ai_piece))
    
    if maxPlayer:
        value = -np.inf
        column = random.choice(columns)
        
        for col in columns:
            for r in range(m):
                if matrix[r][col] == 0:
                 row = r
                 break

            matrix_copy = matrix.copy()
            matrix_copy[row][col]=ai_piece
            
            new_score = rey_minimax(matrix_copy, depth-1, alpha, beta, False, ai_piece, player_piece)[1]
            
            if new_score > value:
                value = new_score
                column = col
            
            alpha = max(alpha, value)
            if alpha >= beta:
                break
        
        return column, value
    
    else:  # Minplayer
        value = np.inf
        column = random.choice(columns)
        
        for col in columns:
            for r in range(m):
                if matrix[r][col] == 0:
                    row = r
                    break


            matrix_copy = matrix.copy()
            matrix_copy[row][col]=player_piece
            
            new_score = rey_minimax(matrix_copy, depth-1, alpha, beta, True, ai_piece, player_piece)[1]
            
            if new_score < value:
                value = new_score
                column = col
            
            beta = min(beta, value)
            if alpha >= beta:
                break
        
        return column, value







def rey_winning_move(A, piece):
   
    for c in range(n-3):
        for r in range(m):
            if A[r][c] == piece and A[r][c+1] == piece and A[r][c+2] == piece and A[r][c+3] == piece:
                return True

   
    for c in range(n):
        for r in range(m-3):
            if A[r][c] == piece and A[r+1][c] == piece and A[r+2][c] == piece and A[r+3][c] == piece:
                return True

    
    for c in range(n-3):
        for r in range(m-3):
            if A[r][c] == piece and A[r+1][c+1] == piece and A[r+2][c+2] == piece and A[r+3][c+3] == piece:
                return True

   
    for c in range(n-3):
        for r in range(3, m):
            if A[r][c] == piece and A[r-1][c+1] == piece and A[r-2][c+2] == piece and A[r-3][c+3] == piece:
                return True
    
    return False




def rey_computeHuristic(p, o, z):
    huristic = 0
   
    if p == 4: huristic += 10000
    elif p == 3 and z == 1: huristic += 100
    elif p == 2 and z == 2: huristic += 10
    
    
    if o == 3 and z == 1: huristic -= 5000 
    elif o == 2 and z == 2: huristic -= 50
    return huristic






def rey_get_next_open_row(matrix, col):
	for r in range(m):
		if matrix[r][col] == 0:
			return r





def rey_counting_cells(matrix, piece):
    count = 0
    
    if piece == bot:
        opponent = human
    else:
       opponent= bot

    

   
    center_array = matrix[:, n//2]
    count += np.count_nonzero(center_array == piece) * 3

    
    for r in range(m):
        for c in range(n-3):
            p, o, z = 0, 0, 0
            for i in range(4):
                val = matrix[r][c+i]
                if val == piece: p += 1
                elif val == opponent: o += 1
                else: z += 1
            count += rey_computeHuristic(p, o, z)

    
    for c in range(n):
        for r in range(m-3):
            p, o, z = 0, 0, 0
            for i in range(4):
                val = matrix[r+i][c]
                if val == piece: p += 1
                elif val == opponent: o += 1
                else: z += 1
            count += rey_computeHuristic(p, o, z)

   
    for r in range(m-3):
        for c in range(n-3):
            p, o, z = 0, 0, 0
            for i in range(4):
                val = matrix[r+i][c+i]
                if val == piece: p += 1
                elif val == opponent: o += 1
                else: z += 1
            count += rey_computeHuristic(p, o, z)

   
    for r in range(3, m):
        for c in range(n-3):
            p, o, z = 0, 0, 0
            for i in range(4):
                val = matrix[r-i][c+i]
                if val == piece: p += 1
                elif val == opponent: o += 1
                else: z += 1
            count += rey_computeHuristic(p, o, z)

    return count



def rey_is_terminal_node(matrix):
    return rey_winning_move(matrix, human) or rey_winning_move(matrix, bot) or len(rey_possibleCols(matrix)) == 0






In [ ]:
import json, websockets, asyncio

async def bot_vs_bot():
    uri = f"wss://maziar-temp-project.ir/connect4/ws/api/v1/game/{created_game_uuid}"

    async with websockets.connect(uri) as ws:
        print("WebSocket connected")
        session.patch(f"{base_url}/api/v1/game/join?game_uuid={created_game_uuid}", headers={"Authorization": opponent_token})

        print("You have 20 sec if you want to go open the browser to see the game")
        await asyncio.sleep(20) 

        msg = json.loads(await ws.recv())
        if msg.get("last_move"):
            print(f"WS: {msg['current_turn']} chose column {msg.get("last_move")}")

        while True:

            status = msg["status"]

            if status == "FINISHED":
                print(f"Game finished. winner: {msg["winner"]} | end_reason: {msg["end_reason"]}")
                break

            current_turn = msg.get("current_turn_id")
            if current_turn == my_id: 
                my_make_move(my_token, created_game_uuid, msg.get("last_move"))
            elif current_turn == opponent_id:    
                opponent_make_move(opponent_token, created_game_uuid, msg.get("last_move"))
            
            msg = json.loads(await ws.recv())
    

await bot_vs_bot()
